# RAG system for web data using ollama service with llama3.2:1b

## Setup

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 12.7 ms, sys: 8.97 ms, total: 21.7 ms
Wall time: 1.02 s


### OSS libraries install

In [2]:
%pip install urllib3 beautifulsoup4 sentence_transformers chromadb huggingface langchain langchain_chroma langchain-community langchain-huggingface unstructured langchain-ollama ollama ipython-autotime

Note: you may need to restart the kernel to use updated packages.


## Index the URLs to create the knowledge base

In [3]:
URLS_DICTIONARY = {
    "gcp": "https://cloud.google.com/docs/overview",
    "azure": "https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure",
    "aws": "https://aws.amazon.com/what-is-aws/",
    "aws_rag_page": "https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/",
    "azure_rag_page": "https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652",
}
COLLECTION_NAME = "my_collection"
documents = []

In [4]:
class Document:
    def __init__(self, metadata, page_content):
        self.metadata = metadata
        self.page_content = page_content

In [5]:
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning
from bs4 import BeautifulSoup
import re

# Suppress SSL warnings for development
urllib3.disable_warnings(InsecureRequestWarning)


def read_url_clean_text(url: str, verify_ssl: bool = False) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    try:
        # Try with SSL verification first
        response = requests.get(url, headers=headers, verify=verify_ssl, timeout=30)
        response.raise_for_status()
    except requests.exceptions.SSLError:
        print(f"SSL verification failed for {url}, trying without verification...")
        # Fallback without SSL verification
        response = requests.get(url, headers=headers, verify=False, timeout=30)
        response.raise_for_status()

    # Parse HTML and extract clean text
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove unwanted elements
    for element in soup(
        ["script", "style", "nav", "header", "footer", "aside", "noscript"]
    ):
        element.decompose()

    # Get text content
    text = soup.get_text()

    # Clean up whitespace and formatting
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = " ".join(chunk for chunk in chunks if chunk)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
for name, url in URLS_DICTIONARY.items():
    print(f"Loading from {url}")
    response = read_url_clean_text(url, verify_ssl=False)

    if len(response) > 0:
        data = {
            "metadata": {"source": url, "name": name},
            "page_content": response,
        }

        documents.append(
            Document(metadata=data["metadata"], page_content=data["page_content"])
        )
        print(f"Loaded from {url}")
    else:
        print(f"Failed to retrieve content from {url}")


print(documents[0].metadata)
print(documents[0].page_content)

Loading from https://cloud.google.com/docs/overview
Loaded from https://cloud.google.com/docs/overview
Loading from https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure
Loaded from https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure
Loading from https://aws.amazon.com/what-is-aws/
Loaded from https://aws.amazon.com/what-is-aws/
Loading from https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/
Loaded from https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/
Loading from https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652
Loaded from https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652
{'source': 'https://cloud.google.com/docs

In [7]:
len(documents)

5

In [8]:
documents[0].metadata

{'source': 'https://cloud.google.com/docs/overview', 'name': 'gcp'}

In [9]:
doc_id = 0
for doc in documents:
    doc.page_content = " ".join(doc.page_content.split())  # remove white space

    doc.metadata["id"] = (
        doc_id  # make a document id and add it to the document metadata
    )

    print(doc.metadata)
    doc_id += 1

# Let's see how our sample document looks now after we cleaned it up.
display(documents[1].metadata)
display(documents[1].page_content)

{'source': 'https://cloud.google.com/docs/overview', 'name': 'gcp', 'id': 0}
{'source': 'https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure', 'name': 'azure', 'id': 1}
{'source': 'https://aws.amazon.com/what-is-aws/', 'name': 'aws', 'id': 2}
{'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page', 'id': 3}
{'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652', 'name': 'azure_rag_page', 'id': 4}


{'source': 'https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure',
 'name': 'azure',
 'id': 1}

'What is Azure? | Microsoft Azure This is the Trace Id: 3dccecc454254f0eec3259c76090e976 Skip to main content What is Azure? Azure is a trusted cloud platform for building, deploying, and managing innovative solutions. Get to know Azure Get started with Azure What is Microsoft Azure? Microsoft Azure, launched in 2010, marked a pivotal shift from on-premises datacenters to cloud computing. By offering businesses a global network of datacenters maintained and managed by Microsoft, Azure reduced the time and expense associated with maintaining on-premises infrastructure. Since its original launch, Azure continues to offer extensive capabilities that go beyond simplifying infrastructure management. With comprehensive AI, data, and application services that work together, Azure delivers a unified approach to cloud computing that’s unique in the industry. Its open, flexible cloud platform is designed to support each company’s business strategy and stage of AI transformation. Key takeaways Az

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=0)
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': 'https://cloud.google.com/docs/overview', 'name': 'gcp', 'id': 0}, page_content="Google Cloud overview | Get started Skip to main content / English Deutsch Español – América Latina Français Indonesia Italiano Português – Brasil 中文 – 简体 中文 – 繁體 日本語 한국어 Console Sign in Get started Contact Us Start free Home Documentation Get started Send feedback Google Cloud overview Stay organized with collections Save and categorize content based on your preferences. This overview is designed to help you understand the overall landscape of Google Cloud. Here, you'll take a brief look at some of the"),
 Document(metadata={'source': 'https://cloud.google.com/docs/overview', 'name': 'gcp', 'id': 0}, page_content="commonly used features and get pointers to documentation that can help you go deeper. Knowing what's available and how the parts work together can help you make decisions about how to proceed. You'll also get pointers to some tutorials that you can use to try out Go

### HuggingFace Embeddings

In [11]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Initialize HuggingFace embeddings with a popular model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Vector Store
Let's load the content into a local instance of a vector database, using Chromadb.

In [12]:
from langchain.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
vectorstore

## Query 
Let's do a quick search of our vector database to test it out!

In [13]:
test1_query = "What is Sagemaker performance?"

test1_search_result = vectorstore.similarity_search_with_score(test1_query, k=4)
test1_search_result

[(Document(metadata={'source': 'https://aws.amazon.com/blogs/machine-learning/optimize-rag-in-production-environments-using-amazon-sagemaker-jumpstart-and-amazon-opensearch-service/', 'name': 'aws_rag_page', 'id': 3}, page_content='to a SageMaker real-time endpoint: sagemaker.jumpstart.model JumpStartModel model_id "meta-textgeneration-llama-3-8b-instruct" accept_eula model JumpStartModel(model_idmodel_id) llm_predictor modeldeploy(accept_eulaaccept_eula) model_id "huggingface-sentencesimilarity-bge-large-en-v1-5" text_embedding_model JumpStartModel(model_idmodel_id) embedding_predictor text_embedding_modeldeploy() Content handlers are crucial for formatting data for SageMaker endpoints. They transform inputs into the format expected'),
  0.9224939346313477),
 (Document(metadata={'name': 'aws', 'source': 'https://aws.amazon.com/what-is-aws/', 'id': 2}, page_content='AWS pioneered the serverless computing space with the launch of AWS Lambda, which lets developers run their code without 

In [14]:
test2_query = "Share examples of agentic RAG?"

test2_search_result = vectorstore.similarity_search_with_score(test2_query, k=4)
test2_search_result

[(Document(metadata={'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652', 'id': 4, 'name': 'azure_rag_page'}, page_content='generated, considering that there’s missing information searches couldn’t find. The agentic RAG loop continues until the answer is of sufficient quality or too much time has passed. Single-Step Reflection We can put all the components of agentic RAG together into our first sample implementation: single-step reflection. The single-shot RAG flow is run to get a candidate answer. The answer is evaluated using relevance and groundedness evaluators. If both scores from these evaluators are at least 4, the'),
  0.8130398988723755),
 (Document(metadata={'id': 4, 'name': 'azure_rag_page', 'source': 'https://techcommunity.microsoft.com/blog/azure-ai-services-blog/bonus-rag-time-journey-agentic-rag/4404652'}, page_content="agents to make a dynamic and self-refining retrieval system. What we'll cover: Overvie

In [15]:
test3_query = "Compare AWS, GCP and Azure"
test3_search_result = vectorstore.similarity_search_with_score(test3_query, k=4)
test3_search_result

[(Document(metadata={'name': 'aws', 'source': 'https://aws.amazon.com/what-is-aws/', 'id': 2}, page_content='The leading cloud Most functionality AWS has significantly more services, and more features within those services, than any other cloud provider–from infrastructure technologies like compute, storage, and databases–to emerging technologies, such as machine learning and artificial intelligence, data lakes and analytics, and Internet of Things. This makes it faster, easier, and more cost effective to move your existing applications to the cloud and build nearly anything you can imagine.AWS also has the'),
  0.8563166260719299),
 (Document(metadata={'name': 'azure', 'id': 1, 'source': 'https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-azure'}, page_content='hardware and firmware components. Built -in resiliency features support high availability, disaster recovery and backup. Azure also supports more than 100 compliance standards. 04/ What is the differ

## Set up a retriever

The retrieved information from the vector store serves as additional context or knowledge that can be used by a generative model. Lets specify search kwargs like k (the number of documents to return (Default: 4)) to use when doing retrieval.

In [16]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x347accd10>, search_kwargs={'k': 4})

## Generate a response with a generative model

Finally, we’ll generate a response. The generative model (llama3.2:1b) uses the retrieved information to produce a more accurate and contextually relevant response to the questions.

## LLM model construction

In [17]:
from langchain_ollama.llms import OllamaLLM

my_ollama_model = "llama3.2"

llm_client = OllamaLLM(
    model=my_ollama_model,
    base_url="http://localhost:11434",
    headers={"Content-Type": "application/json"},
    stream=True,
)

llm_client

OllamaLLM(model='llama3.2', base_url='http://localhost:11434')

### Using prompt template

In [18]:
from langchain_core.prompts import ChatPromptTemplate

# Create a ChatPromptTemplate for the RAG system
template = """Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. 
Answer style should match the context. Ideal Answer Length 2-10 sentences.\n\n{context}\nQuestion: {question}\nAnswer:
"""

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant that answers questions based on the provided context. Use only the information from the context to answer the question. If the context doesn't contain enough information to answer the question, say so.",
        ),
        ("human", "Context: {context}\n\nQuestion: {question}"),
    ]
)

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. \nAnswer style should match the context. Ideal Answer Length 2-10 sentences.\n\n{context}\nQuestion: {question}\nAnswer:\n'), additional_kwargs={})])

In [19]:
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

### Integrating LangChain

In [20]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Create the RAG chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm_client
    | StrOutputParser()
)

chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x347accd10>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Generate a summary of the context that answers the question. Explain the answer in multiple steps if possible. \nAnswer style should match the context. Ideal Answer Length 2-10 sentences.\n\n{context}\nQuestion: {question}\nAnswer:\n'), additional_kwargs={})])
| OllamaLLM(model='llama3.2', base_url='http://localhost:11434')
| StrOutputParser()

## TEST to ask Questions

In [21]:
print("Test question: ", test1_query)

test1_response = chain.invoke(test1_query)

print("Test1 response: ", test1_response)

Test question:  What is Sagemaker performance?
Test1 response:  The performance of Amazon SageMaker refers to its ability to provide a scalable, secure, and reliable platform for building, training, and deploying machine learning models. Here's a summary of the context:

SageMaker JumpStart has streamlined the process of developing and deploying generative AI applications by offering pre-trained models, user-friendly interfaces, and seamless scalability within the AWS ecosystem.

Key benefits of SageMaker performance include:
- **Unmatched experience**: SageMaker leverages the operational expertise and maturity of AWS to provide a dependable solution for critical applications.
- **Security**: SageMaker ensures the security and integrity of machine learning data and models, protecting against unauthorized access and tampering.
- **Performance**: SageMaker provides high-performance computing resources and optimized algorithms for training and deploying machine learning models, ensuring f

In [22]:
import pprint

pprint.pprint(test1_response)

('The performance of Amazon SageMaker refers to its ability to provide a '
 'scalable, secure, and reliable platform for building, training, and '
 "deploying machine learning models. Here's a summary of the context:\n"
 '\n'
 'SageMaker JumpStart has streamlined the process of developing and deploying '
 'generative AI applications by offering pre-trained models, user-friendly '
 'interfaces, and seamless scalability within the AWS ecosystem.\n'
 '\n'
 'Key benefits of SageMaker performance include:\n'
 '- **Unmatched experience**: SageMaker leverages the operational expertise '
 'and maturity of AWS to provide a dependable solution for critical '
 'applications.\n'
 '- **Security**: SageMaker ensures the security and integrity of machine '
 'learning data and models, protecting against unauthorized access and '
 'tampering.\n'
 '- **Performance**: SageMaker provides high-performance computing resources '
 'and optimized algorithms for training and deploying machine learning models, '

In [23]:
print("Test question: ", test2_query)

test2_response = chain.invoke(test2_query)

print("Test2 response: ", test2_response)

Test question:  Share examples of agentic RAG?
Test2 response:  Here's a summary of the context that answers the question about Agentic RAG:

Agentic RAG (Reinforcement And Generative Loop) is an AI framework that leverages generative models to enable agents to make decisions and execute actions autonomously. It improves traditional RAG flow by actively interacting with its environment using tools, memory, and secure access to data.

To illustrate this concept, we can break it down into three key examples of agentic RAG:

1. **Single-Step Reflection**: This involves running a single-shot RAG flow to generate a candidate answer. The answer is then evaluated using relevance and groundedness evaluators. If both scores are at least 4, the answer is considered satisfactory.
2. **Multi-Step Reflection**: This example builds upon the single-step reflection by introducing multiple iterations of the RAG flow. Each iteration refines the generated answer based on feedback from evaluators, leading

In [24]:
pprint.pprint(test2_response)

("Here's a summary of the context that answers the question about Agentic "
 'RAG:\n'
 '\n'
 'Agentic RAG (Reinforcement And Generative Loop) is an AI framework that '
 'leverages generative models to enable agents to make decisions and execute '
 'actions autonomously. It improves traditional RAG flow by actively '
 'interacting with its environment using tools, memory, and secure access to '
 'data.\n'
 '\n'
 'To illustrate this concept, we can break it down into three key examples of '
 'agentic RAG:\n'
 '\n'
 '1. **Single-Step Reflection**: This involves running a single-shot RAG flow '
 'to generate a candidate answer. The answer is then evaluated using relevance '
 'and groundedness evaluators. If both scores are at least 4, the answer is '
 'considered satisfactory.\n'
 '2. **Multi-Step Reflection**: This example builds upon the single-step '
 'reflection by introducing multiple iterations of the RAG flow. Each '
 'iteration refines the generated answer based on feedback from ev

In [25]:
print("Test question: ", test3_query)

test3_response = chain.invoke(test3_query)

test3_response

Test question:  Compare AWS, GCP and Azure


"Comparing the leading cloud providers AWS (Amazon Web Services), GCP (Google Cloud Platform), and Azure is a complex task due to their extensive offerings. However, here's a summary of the context:\n\nEach provider excels in various areas, making it challenging to declare a single winner. AWS has significantly more services and features than its competitors, with a broader range of infrastructure technologies, emerging technologies like machine learning and AI, data lakes and analytics, and IoT. This makes it an attractive choice for businesses seeking to move their existing applications to the cloud or build custom solutions.\n\nAzure stands out for its extensive global network of datacenters (over 400 in over 70 regions), delivering a more extensive cloud footprint than GCP. Azure also offers built-in resiliency features, support for more than 100 compliance standards, and is trusted by 95% of Fortune 500 companies.\n\nGCP, on the other hand, excels in areas like machine learning, d

In [26]:
pprint.pprint(test3_response)

('Comparing the leading cloud providers AWS (Amazon Web Services), GCP (Google '
 'Cloud Platform), and Azure is a complex task due to their extensive '
 "offerings. However, here's a summary of the context:\n"
 '\n'
 'Each provider excels in various areas, making it challenging to declare a '
 'single winner. AWS has significantly more services and features than its '
 'competitors, with a broader range of infrastructure technologies, emerging '
 'technologies like machine learning and AI, data lakes and analytics, and '
 'IoT. This makes it an attractive choice for businesses seeking to move their '
 'existing applications to the cloud or build custom solutions.\n'
 '\n'
 'Azure stands out for its extensive global network of datacenters (over 400 '
 'in over 70 regions), delivering a more extensive cloud footprint than GCP. '
 'Azure also offers built-in resiliency features, support for more than 100 '
 'compliance standards, and is trusted by 95% of Fortune 500 companies.\n'
 '\n'
 